In [ ]:
#角点检测

import os
import cv2
import numpy as np
import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False


output_dir = "biaoding/result"
os.makedirs(output_dir, exist_ok=True)  

# 可视化角点检测
img_path = r"biaoding2\fluo\1.jpg"  
img = cv2.imread(img_path)

if img is None:
    print(f"无法读取图像: {img_path}")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, (9,6), None)

if ret:
    # 绘制角点
    cv2.drawChessboardCorners(img, (9,6), corners, ret)
    
    # 创建保存路径
    base_name = os.path.basename(img_path) 
    save_path = os.path.join(output_dir, f"corners_{base_name}") 
    
    # 保存图像
    cv2.imwrite(save_path, img) 
    print(f"已保存角点检测结果到: {save_path}")
    
    # 显示图像
    cv2.namedWindow("Corners", cv2.WINDOW_NORMAL)
    cv2.imshow("Corners", img)
    
    # 等待按键后调整窗口大小
    key = cv2.waitKey(0)
    if key == ord('r'):  # 按下 'r' 键调整窗口大小
        cv2.resizeWindow("Corners", 1024, 768)
    
    # 关闭窗口
    cv2.destroyAllWindows()

else:
    print("请检查：1. pattern_size 2. 光照条件 3. 棋盘格平整度")
    save_path_fail = os.path.join(output_dir, f"failed_{os.path.basename(img_path)}")
    cv2.imwrite(save_path_fail, img)
    print(f"已保存失败检测的图像到: {save_path_fail}")

已保存角点检测结果到: biaoding/result\corners_1.jpg


In [ ]:
#配准

In [ ]:
import cv2
import numpy as np
import glob
import os
import time
import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False  

def calibrate_cameras(rgb_dir, fluo_dir, pattern_size=(9,6)):
    start_time = time.time()
    print("开始...")
    
    # 定义标定参数
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    objp = np.zeros((pattern_size[0]*pattern_size[1],3), np.float32)
    objp[:,:2] = np.mgrid[0:pattern_size[0],0:pattern_size[1]].T.reshape(-1,2)

    # 检测角点
    def find_corners(image_paths, is_fluo=False):
        obj_points = []
        img_points = []
        valid_paths = []
        
        for i, path in enumerate(image_paths):

            if is_fluo:
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"无法加载荧光图像: {path}")
                    continue
                # 增强荧光图像对比度
                img = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(img)
            else:
                img = cv2.imread(path)
                if img is None:
                    print(f"无法加载RGB图像: {path}")
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            print(f"处理图像 {i+1}/{len(image_paths)}: {os.path.basename(path)}")
            ret, corners = cv2.findChessboardCorners(img, pattern_size, None)
            if ret:
                # 角点细化
                corners_refined = cv2.cornerSubPix(img, corners, (11,11), (-1,-1), criteria)
                obj_points.append(objp)
                img_points.append(corners_refined)
                valid_paths.append(path)
                print(f"✓ 成功检测到角点")
            else:
                print(f"✗ 未检测到角点")
                
        return obj_points, img_points, valid_paths

    # 获取图像路径
    rgb_images = sorted(glob.glob(os.path.join(rgb_dir, "*.jpg")))
    fluo_images = sorted(glob.glob(os.path.join(fluo_dir, "*.png")))
    
    # 确保图像数量一致
    min_count = min(len(rgb_images), len(fluo_images))
    rgb_images = rgb_images[:min_count]
    fluo_images = fluo_images[:min_count]
    
    print(f"找到 {min_count} 对图像用于配准")
    
    # 检测角点
    print("\n检测RGB图像角点...")
    rgb_objp, rgb_imgp, rgb_valid = find_corners(rgb_images)
    print("\n检测荧光图像角点...")
    fluo_objp, fluo_imgp, fluo_valid = find_corners(fluo_images, is_fluo=True)
    
    if not rgb_imgp or not fluo_imgp:
        raise ValueError("未能找到足够的棋盘格角点，请检查图像目录或文件路径以及pattern_size")
    
    # 确保配对正确
    if len(rgb_imgp) != len(fluo_imgp):
        print("警告: RGB和荧光图像成功检测的角点数量不匹配")
        # 使用最小数量
        min_pairs = min(len(rgb_imgp), len(fluo_imgp))
        rgb_imgp = rgb_imgp[:min_pairs]
        fluo_imgp = fluo_imgp[:min_pairs]
    
    # 使用所有检测到的角点计算单应矩阵
    all_rgb_points = np.vstack(rgb_imgp).reshape(-1, 2)
    all_fluo_points = np.vstack(fluo_imgp).reshape(-1, 2)
    
    # RANSAC计算单应矩阵
    print("计算单应矩阵...")
    H, mask = cv2.findHomography(all_rgb_points, all_fluo_points, cv2.RANSAC, 5.0)
    
    # 计算内点比例评估配准质量
    inlier_ratio = np.sum(mask) / len(mask) if mask is not None else 0
    print(f"单应矩阵计算完成，使用 {len(rgb_imgp)} 对图像，内点比例: {inlier_ratio:.2%}")
    
    elapsed_time = time.time() - start_time
    print(f"标定完成，耗时: {elapsed_time:.2f}秒")
    
    return H, rgb_imgp, fluo_imgp

def apply_registration(rgb_path, fluo_path, H, output_dir=None):
    """应用单应矩阵进行图像配准"""
    # 读取图像
    rgb_img = cv2.imread(rgb_path)
    fluo_img = cv2.imread(fluo_path, cv2.IMREAD_GRAYSCALE)
    
    if rgb_img is None or fluo_img is None:
        print(f"无法读取图像: {rgb_path} 或 {fluo_path}")
        return None
    
    # 应用单应矩阵变换
    registered_rgb = cv2.warpPerspective(rgb_img, H, (fluo_img.shape[1], fluo_img.shape[0]))
    
    # 创建叠加图像
    fluo_color = cv2.cvtColor(fluo_img, cv2.COLOR_GRAY2BGR)
    overlay = cv2.addWeighted(registered_rgb, 0.7, fluo_color, 0.3, 0)
    
    # 保存结果
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(rgb_path)
        
        # 保存配准后的RGB图像
        cv2.imwrite(os.path.join(output_dir, f"registered_{base_name}"), registered_rgb)
        
        # 保存叠加图像
        cv2.imwrite(os.path.join(output_dir, f"overlay_{base_name}"), overlay)
        
        print(f"已保存配准结果: {base_name}")
    
    return registered_rgb, overlay

def batch_registration(rgb_dir, fluo_dir, output_dir, pattern_size=(9,6)):
    """批量处理所有图像对"""
    start_time = time.time()
    print("开始批量配准...")
    
    # 计算单应矩阵
    H, rgb_imgp, fluo_imgp = calibrate_cameras(rgb_dir, fluo_dir, pattern_size)
    print("标定得到的单应矩阵H:\n", H)
    
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取所有RGB图像路径
    rgb_images = sorted(glob.glob(os.path.join(rgb_dir, "*.jpg")))
    
    # 处理每对图像
    total_images = len(rgb_images)
    processed_count = 0
    
    for i, rgb_path in enumerate(rgb_images):
        base_name = os.path.basename(rgb_path)
        base_name_no_ext = os.path.splitext(base_name)[0] 
        
        # 构建荧光图像路径
        fluo_path = os.path.join(fluo_dir, f"{base_name_no_ext}.png")
        
        if not os.path.exists(fluo_path):
            print(f"找不到对应的荧光图像: {base_name}")
            continue
        
        print(f"\n处理图像对 {i+1}/{total_images}: {base_name}")
        start_pair_time = time.time()
        
        # 应用配准
        apply_registration(rgb_path, fluo_path, H, output_dir)
        
        pair_time = time.time() - start_pair_time
        print(f"完成处理，耗时: {pair_time:.2f}秒")
        processed_count += 1
    
    total_time = time.time() - start_time
    print(f"\n所有图像配准完成，共处理 {processed_count}/{total_images} 对图像")
    print(f"总耗时: {total_time:.2f}秒，结果保存在: {output_dir}")
    return H, rgb_imgp, fluo_imgp

# 配置参数
RGB_DIR = "biaoding2/rgb"  # RGB图像目录
FLUO_DIR = "biaoding2/fluo"  # 荧光图像目录
OUTPUT_DIR = "biaoding/result"  # 输出目录
PATTERN_SIZE = (9, 6)  # 棋盘格内角点数

print("="*50)
print("图像配准程序启动")
print(f"RGB目录: {RGB_DIR}")
print(f"荧光目录: {FLUO_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"棋盘格参数: {PATTERN_SIZE}")
print("="*50)

# 执行批量配准并获取角点数据
H, rgb_imgp, fluo_imgp = batch_registration(RGB_DIR, FLUO_DIR, OUTPUT_DIR, PATTERN_SIZE)

# 保存单应矩阵供后续使用
np.save(os.path.join(OUTPUT_DIR, "homography_matrix.npy"), H)
print(f"单应矩阵已保存到: {os.path.join(OUTPUT_DIR, 'homography_matrix.npy')}")

# 可视化第一对图像的配准结果
rgb_images = sorted(glob.glob(os.path.join(RGB_DIR, "*.jpg")))
fluo_images = sorted(glob.glob(os.path.join(FLUO_DIR, "*.png")))

if rgb_images and fluo_images:
    print("\n可视化第一对图像...")
    rgb_img = cv2.imread(rgb_images[0])
    fluo_img = cv2.imread(fluo_images[0], cv2.IMREAD_GRAYSCALE)
    
    # 应用配准
    registered_rgb, overlay = apply_registration(rgb_images[0], fluo_images[0], H)
    
    # 创建可视化
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f"图像配准结果: {os.path.basename(rgb_images[0])}", fontsize=16)
    
    # 原始RGB图像
    axs[0, 0].imshow(cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB))
    axs[0, 0].set_title("原始RGB图像")
    axs[0, 0].axis('off')
    
    # 原始荧光图像
    axs[0, 1].imshow(fluo_img, cmap='gray')
    axs[0, 1].set_title("原始荧光图像")
    axs[0, 1].axis('off')
    
    # 配准后的RGB图像
    axs[1, 0].imshow(cv2.cvtColor(registered_rgb, cv2.COLOR_BGR2RGB))
    axs[1, 0].set_title("配准后的RGB图像")
    axs[1, 0].axis('off')
    
    # 叠加结果
    axs[1, 1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axs[1, 1].set_title("RGB与荧光叠加")
    axs[1, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "registration_visualization.png"), dpi=300, bbox_inches='tight')
    print(f"配准可视化已保存到: {os.path.join(OUTPUT_DIR, 'registration_visualization.png')}")
    plt.show()
